In [ ]:
# ============================================================
# CELL 1 — GLOBAL PARAMETERS (ADJUST EVERYTHING HERE)
# ============================================================

# --- Fundamental constants ---
mu0 = 4 * np.pi * 1e-7          # magnetic constant

# --- Torus geometry ---
R_major = 0.3                   # major radius (m)
r_minor = 0.1                   # minor radius (m)

# --- Coil currents ---
coil_current = 5000             # toroidal coil current (A)
nozzle_current = 3000           # nozzle coil current (A)

# --- Nozzle geometry ---
nozzle_offset_x = 0.05          # nozzle position relative to torus (m)
nozzle_offset_z = -0.05         # downward flare (m)
nozzle_radius = 0.05            # nozzle coil radius (m)
nozzle_segments = 100           # resolution

# --- Coil discretization ---
coil_segments = 200             # resolution of toroidal coil

# --- Field sampling ---
field_sample_points = 2000      # number of points inside torus for field viz

# --- Switching gate ---
f_switch = 5.0                  # gate frequency (Hz)
containment_open_factor = 0.5   # containment strength when gate is open

# --- Particle properties ---
q_over_m = 5e5                  # charge-to-mass ratio
particle_mass = 1e-12           # kg (toy mass)

# --- Integration parameters ---
dt = 1e-5                       # timestep
n_steps_single = 3000           # steps for single packet
n_steps_multi = 2000            # steps for multi-packet

# --- Multi-packet plume ---
num_packets = 500               # number of packets in plume
packet_position_jitter = 0.01   # random offset around gate
packet_velocity_jitter = 20     # random velocity variation

# --- Plotting ---
sphere_radius = 0.01            # size of trajectory dots
max_clip_radius = 0.8           # clip runaway trajectories

print("Cell 1 loaded: all global parameters set.")


In [ ]:
# ============================================================
# CELL 2 — GEOMETRY + COIL CONSTRUCTION
# ============================================================

import numpy as np
import pyvista as pv

# ------------------------------------------------------------
# Plasma torus geometry
# ------------------------------------------------------------
plasma_torus = pv.ParametricTorus(R_major, r_minor)

# ------------------------------------------------------------
# Coil torus (visual only)
# ------------------------------------------------------------
coil_torus = pv.ParametricTorus(R_major, r_minor * 0.4)

# ------------------------------------------------------------
# Sample points inside torus for field visualization
# ------------------------------------------------------------
theta = np.random.uniform(0, 2*np.pi, field_sample_points)
phi   = np.random.uniform(0, 2*np.pi, field_sample_points)

x = (R_major + r_minor * np.cos(phi)) * np.cos(theta)
y = (R_major + r_minor * np.cos(phi)) * np.sin(theta)
z = r_minor * np.sin(phi)

points = np.column_stack((x, y, z))

# ------------------------------------------------------------
# Toroidal coil discretization
# ------------------------------------------------------------
coil_theta = np.linspace(0, 2*np.pi, coil_segments)

coil_x = (R_major + r_minor * 0.4 * np.cos(coil_theta)) * np.cos(coil_theta)
coil_y = (R_major + r_minor * 0.4 * np.cos(coil_theta)) * np.sin(coil_theta)
coil_z = r_minor * 0.4 * np.sin(coil_theta)

coil_loop = np.column_stack((coil_x, coil_y, coil_z))
coil_dl   = np.roll(coil_loop, -1, axis=0) - coil_loop

# ------------------------------------------------------------
# Nozzle coil geometry
# ------------------------------------------------------------
nozzle_center = np.array([
    R_major + nozzle_offset_x,
    0.0,
    nozzle_offset_z
])

nozzle_theta = np.linspace(0, 2*np.pi, nozzle_segments)

nozzle_x = nozzle_center[0] + nozzle_radius * np.cos(nozzle_theta)
nozzle_y = nozzle_center[1] + nozzle_radius * np.sin(nozzle_theta)
nozzle_z = np.full_like(nozzle_x, nozzle_center[2])

nozzle_loop = np.column_stack((nozzle_x, nozzle_y, nozzle_z))
nozzle_dl   = np.roll(nozzle_loop, -1, axis=0) - nozzle_loop
nozzle_mid  = nozzle_loop.copy()

print("Cell 2 loaded: geometry, coils, nozzle, and field sample points ready.")


In [ ]:
# ============================================================
# CELL 3 — MAGNETIC FIELD FUNCTIONS
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Gate timing (open/closed)
# ------------------------------------------------------------
def gate_phase(t):
    return (t * f_switch) % 1.0

# ------------------------------------------------------------
# Nozzle current switching
# ------------------------------------------------------------
def nozzle_current_t(t):
    # Gate open for first half of each cycle
    return nozzle_current if gate_phase(t) < 0.5 else 0.0

# ------------------------------------------------------------
# Containment scaling (half power when gate open)
# ------------------------------------------------------------
def containment_factor_t(t):
    return containment_open_factor if gate_phase(t) < 0.5 else 1.0

# ------------------------------------------------------------
# Toroidal coil magnetic field (Biot–Savart)
# ------------------------------------------------------------
def B_field_toroidal(x, y, z):
    r_vec = np.array([x, y, z])
    diff = r_vec - coil_loop
    dist = np.linalg.norm(diff, axis=1)
    mask = dist > 1e-6

    dB = mu0 * coil_current / (4*np.pi) * np.cross(
        coil_dl[mask],
        diff[mask]
    ) / (dist[mask]**3)[:, None]

    return dB.sum(axis=0)

# ------------------------------------------------------------
# Nozzle coil magnetic field (Biot–Savart)
# ------------------------------------------------------------
def B_field_nozzle(x, y, z, t):
    I_t = nozzle_current_t(t)
    if I_t == 0.0:
        return np.zeros(3)

    r_vec = np.array([x, y, z])
    diff = r_vec - nozzle_mid
    dist = np.linalg.norm(diff, axis=1)
    mask = dist > 1e-6

    dB = mu0 * I_t / (4*np.pi) * np.cross(
        nozzle_dl[mask],
        diff[mask]
    ) / (dist[mask]**3)[:, None]

    return dB.sum(axis=0)

# ------------------------------------------------------------
# Combined magnetic field (toroidal + nozzle)
# ------------------------------------------------------------
def B_total(x, y, z, t):
    return (
        containment_factor_t(t) * B_field_toroidal(x, y, z)
        + B_field_nozzle(x, y, z, t)
    )

print("Cell 3 loaded: magnetic field functions ready.")


In [ ]:
# ============================================================
# CELL 4 — RK4 INTEGRATOR (PURE MATH)
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Magnetic field lookup wrapper
# ------------------------------------------------------------
def B_at(r_vec, t):
    # r_vec is a 3‑vector [x, y, z]
    return B_total(r_vec[0], r_vec[1], r_vec[2], t)

# ------------------------------------------------------------
# Lorentz acceleration
# ------------------------------------------------------------
def accel(r_vec, v_vec, t):
    # a = (q/m) * (v × B)
    return q_over_m * np.cross(v_vec, B_at(r_vec, t))

# ------------------------------------------------------------
# RK4 step
# ------------------------------------------------------------
def rk4_step(r_vec, v_vec, t, dt):
    # k1
    a1 = accel(r_vec, v_vec, t)
    k1_v = a1 * dt
    k1_r = v_vec * dt

    # k2
    a2 = accel(r_vec + 0.5 * k1_r, v_vec + 0.5 * k1_v, t + 0.5 * dt)
    k2_v = a2 * dt
    k2_r = (v_vec + 0.5 * k1_v) * dt

    # k3
    a3 = accel(r_vec + 0.5 * k2_r, v_vec + 0.5 * k2_v, t + 0.5 * dt)
    k3_v = a3 * dt
    k3_r = (v_vec + 0.5 * k2_v) * dt

    # k4
    a4 = accel(r_vec + k3_r, v_vec + k3_v, t + dt)
    k4_v = a4 * dt
    k4_r = (v_vec + k3_v) * dt

    # Combine increments
    v_next = v_vec + (k1_v + 2*k2_v + 2*k3_v + k4_v) / 6.0
    r_next = r_vec + (k1_r + 2*k2_r + 2*k3_r + k4_r) / 6.0

    return r_next, v_next

print("Cell 4 loaded: RK4 integrator ready.")


In [ ]:
# ============================================================
# CELL 5 — SINGLE PACKET SIMULATION + PLOT
# ============================================================

import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# ------------------------------------------------------------
# Initial packet position (near nozzle)
# ------------------------------------------------------------
r = np.array([
    R_major + nozzle_offset_x,
    0.0,
    0.0
])

# Initial velocity
v = np.array([300.0, 300.0, 0.0])

t = 0.0
traj = []

# ------------------------------------------------------------
# Integrate trajectory
# ------------------------------------------------------------
for i in range(n_steps_single):
    traj.append(r.copy())
    r, v = rk4_step(r, v, t, dt)
    t += dt

traj = np.array(traj)

# ------------------------------------------------------------
# Clip runaway trajectories
# ------------------------------------------------------------
traj_clipped = traj[np.linalg.norm(traj, axis=1) < max_clip_radius]

# ------------------------------------------------------------
# Compute speeds
# ------------------------------------------------------------
speeds = np.zeros(len(traj_clipped))
if len(traj_clipped) > 1:
    diffs = traj_clipped[1:] - traj_clipped[:-1]
    speeds[1:] = np.linalg.norm(diffs, axis=1)

# ------------------------------------------------------------
# Build PyVista objects
# ------------------------------------------------------------
traj_points = pv.PolyData(traj_clipped)
traj_points["speed"] = speeds

sphere = pv.Sphere(radius=sphere_radius)

traj_line = pv.Spline(traj_clipped, n_points=len(traj_clipped))

# Field snapshot for context
t_mid = 0.5
B_snapshot = np.array([B_total(px, py, pz, t_mid) for px, py, pz in points])
cloud = pv.PolyData(points)
cloud["B"] = B_snapshot

# ------------------------------------------------------------
# Plot everything
# ------------------------------------------------------------
plotter = pv.Plotter()

plotter.add_mesh(plasma_torus, color="cyan", opacity=0.5)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(pv.PolyData(nozzle_loop), color="green", point_size=5)

plotter.add_mesh(
    cloud.glyph(orient="B", scale="B", factor=0.06),
    color="red"
)

plotter.add_mesh(
    traj_points.glyph(scale=False, geom=sphere),
    scalars="speed",
    cmap="coolwarm"
)

plotter.add_mesh(
    traj_line,
    color="white",
    line_width=3
)

plotter.add_axes()
plotter.show_bounds(grid='front', location='outer', all_edges=True)

plotter.camera_position = [
    (0.4, 0.4, 0.4),
    (0.0, 0.0, 0.0),
    (0.0, 0.0, 1.0)
]

plotter.show(title="Single Packet: Speed-Coloured Trajectory + Line")


In [ ]:
# ============================================================
# CELL 6 — SINGLE PACKET THRUST (PLUME DIRECTION PROJECTION)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Ensure we have a valid trajectory
# ------------------------------------------------------------
if len(traj_clipped) < 2:
    print("Not enough trajectory points for thrust estimation.")
else:
    # ------------------------------------------------------------
    # Compute plume direction from start → end displacement
    # ------------------------------------------------------------
    start_pos = traj_clipped[0]
    end_pos   = traj_clipped[-1]

    disp = end_pos - start_pos
    disp_norm = np.linalg.norm(disp)

    if disp_norm == 0:
        print("Plume direction undefined (no net displacement).")
    else:
        n_hat = disp / disp_norm   # unit vector along plume

        # ------------------------------------------------------------
        # Compute velocities from trajectory differences
        # ------------------------------------------------------------
        velocities = np.zeros_like(traj_clipped)
        velocities[1:] = (traj_clipped[1:] - traj_clipped[:-1]) / dt

        # ------------------------------------------------------------
        # Momentum p = m v
        # ------------------------------------------------------------
        momentum = particle_mass * velocities

        # ------------------------------------------------------------
        # Project momentum onto plume direction
        # ------------------------------------------------------------
        p_parallel = momentum @ n_hat

        # ------------------------------------------------------------
        # Momentum flux (dp/dt)
        # ------------------------------------------------------------
        dp_parallel_dt = np.diff(p_parallel) / dt

        thrust_estimate = (
            np.mean(dp_parallel_dt) if len(dp_parallel_dt) > 0 else 0.0
        )

        # ------------------------------------------------------------
        # Print results
        # ------------------------------------------------------------
        print("--------------------------------------------------")
        print(" SINGLE PACKET THRUST (ALONG PLUME DIRECTION)")
        print("--------------------------------------------------")
        print(f"Plume direction (unit vector): {n_hat}")
        print(f"Particle mass: {particle_mass:.3e} kg")
        print(f"Average thrust: {thrust_estimate:.3e} N")
        print("--------------------------------------------------")

        # ------------------------------------------------------------
        # Plot thrust curve
        # ------------------------------------------------------------
        plt.figure(figsize=(10,4))
        plt.plot(dp_parallel_dt)
        plt.title("Momentum Flux Along Plume Direction (dp_parallel/dt)")
        plt.xlabel("Step")
        plt.ylabel("dp_parallel/dt (N)")
        plt.grid(True)
        plt.show()


In [ ]:
# ============================================================
# CELL 7 — MULTI-PACKET PLUME SIMULATION + SUMMED THRUST
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

all_dp_dt = []   # store dp/dt curves for each packet

# ------------------------------------------------------------
# Run multi-packet simulation
# ------------------------------------------------------------
for p in range(num_packets):

    # ------------------------------------------------------------
    # Randomised initial position near nozzle
    # ------------------------------------------------------------
    r = np.array([
        R_major + nozzle_offset_x + np.random.uniform(-packet_position_jitter, packet_position_jitter),
        np.random.uniform(-packet_position_jitter, packet_position_jitter),
        np.random.uniform(-packet_position_jitter, packet_position_jitter)
    ])

    # ------------------------------------------------------------
    # Randomised initial velocity
    # ------------------------------------------------------------
    v = np.array([
        300.0 + np.random.uniform(-packet_velocity_jitter, packet_velocity_jitter),
        300.0 + np.random.uniform(-packet_velocity_jitter, packet_velocity_jitter),
        np.random.uniform(-packet_velocity_jitter, packet_velocity_jitter)
    ])

    t = 0.0
    traj = []

    # ------------------------------------------------------------
    # Integrate packet trajectory
    # ------------------------------------------------------------
    for i in range(n_steps_multi):
        traj.append(r.copy())
        r, v = rk4_step(r, v, t, dt)
        t += dt

    traj = np.array(traj)

    # ------------------------------------------------------------
    # Clip runaway trajectories
    # ------------------------------------------------------------
    traj_clipped = traj[np.linalg.norm(traj, axis=1) < max_clip_radius]
    if len(traj_clipped) < 2:
        continue

    # ------------------------------------------------------------
    # Compute velocities
    # ------------------------------------------------------------
    velocities = np.zeros_like(traj_clipped)
    velocities[1:] = (traj_clipped[1:] - traj_clipped[:-1]) / dt

    # ------------------------------------------------------------
    # Compute plume direction
    # ------------------------------------------------------------
    disp = traj_clipped[-1] - traj_clipped[0]
    disp_norm = np.linalg.norm(disp)
    if disp_norm == 0:
        continue

    n_hat = disp / disp_norm

    # ------------------------------------------------------------
    # Momentum projection
    # ------------------------------------------------------------
    momentum = particle_mass * velocities
    p_parallel = momentum @ n_hat

    # ------------------------------------------------------------
    # Momentum flux dp/dt
    # ------------------------------------------------------------
    dp_parallel_dt = np.diff(p_parallel) / dt

    all_dp_dt.append(dp_parallel_dt)

# ------------------------------------------------------------
# Summed thrust across all packets
# ------------------------------------------------------------
if not all_dp_dt:
    print("No valid packets for thrust estimation.")
else:
    # Pad arrays to equal length
    max_len = max(len(arr) for arr in all_dp_dt)
    padded = np.array([
        np.pad(arr, (0, max_len - len(arr)))
        for arr in all_dp_dt
    ])

    total_dp_dt = padded.sum(axis=0)
    avg_thrust = np.mean(total_dp_dt)

    print("==================================================")
    print(" MULTI-PACKET THRUST RESULT")
    print("==================================================")
    print(f"Packets simulated: {num_packets}")
    print(f"Average thrust along plume direction: {avg_thrust:.3e} N")
    print("==================================================")

    # ------------------------------------------------------------
    # Plot total thrust curve
    # ------------------------------------------------------------
    plt.figure(figsize=(10,4))
    plt.plot(total_dp_dt)
    plt.title("Total Momentum Flux (Summed Across Packets)")
    plt.xlabel("Step")
    plt.ylabel("Total dp/dt (N)")
    plt.grid(True)
    plt.show()


In [ ]:
# ============================================================
# CELL 8 — FULL NOTEBOOK RUNNER
# ============================================================

print("===================================================")
print(" RUNNING FULL PLASMA SLINGSHOT NOTEBOOK (Cells 1→7)")
print("===================================================\n")

# ------------------------------------------------------------
# 1. Load global parameters
# ------------------------------------------------------------
print("Running Cell 1: Global parameters...")
# (Cell 1 must already be executed manually before Cell 8)
# Nothing to run here — parameters are already in memory.

# ------------------------------------------------------------
# 2. Build geometry + coils
# ------------------------------------------------------------
print("Running Cell 2: Geometry + coils...")
# Re-run Cell 2 code
# (We call the functions and definitions directly)
# Geometry objects are already created when Cell 2 was executed.

# ------------------------------------------------------------
# 3. Magnetic field functions
# ------------------------------------------------------------
print("Running Cell 3: Magnetic field functions...")
# These functions are already defined when Cell 3 was executed.

# ------------------------------------------------------------
# 4. RK4 integrator
# ------------------------------------------------------------
print("Running Cell 4: RK4 integrator...")
# RK4 functions are already defined when Cell 4 was executed.

# ------------------------------------------------------------
# 5. Single packet simulation + plot
# ------------------------------------------------------------
print("Running Cell 5: Single packet simulation...")
# Execute Cell 5 code block
# (We simply call the same logic again)
# Re-run the single-packet simulation
r = np.array([R_major + nozzle_offset_x, 0.0, 0.0])
v = np.array([300.0, 300.0, 0.0])
t = 0.0
traj = []
for i in range(n_steps_single):
    traj.append(r.copy())
    r, v = rk4_step(r, v, t, dt)
    t += dt
traj = np.array(traj)
traj_clipped = traj[np.linalg.norm(traj, axis=1) < max_clip_radius]

# ------------------------------------------------------------
# 6. Single packet thrust
# ------------------------------------------------------------
print("Running Cell 6: Single packet thrust...")
if len(traj_clipped) > 1:
    start_pos = traj_clipped[0]
    end_pos   = traj_clipped[-1]
    disp = end_pos - start_pos
    disp_norm = np.linalg.norm(disp)
    if disp_norm > 0:
        n_hat = disp / disp_norm
        velocities = np.zeros_like(traj_clipped)
        velocities[1:] = (traj_clipped[1:] - traj_clipped[:-1]) / dt
        momentum = particle_mass * velocities
        p_parallel = momentum @ n_hat
        dp_parallel_dt = np.diff(p_parallel) / dt
        single_thrust = np.mean(dp_parallel_dt)
        print(f"Single packet thrust: {single_thrust:.3e} N")
    else:
        print("Single packet thrust: plume direction undefined.")
else:
    print("Single packet thrust: insufficient trajectory.")

# ------------------------------------------------------------
# 7. Multi-packet plume + thrust
# ------------------------------------------------------------
print("Running Cell 7: Multi-packet plume simulation...")
all_dp_dt = []
for p in range(num_packets):
    r = np.array([
        R_major + nozzle_offset_x + np.random.uniform(-packet_position_jitter, packet_position_jitter),
        np.random.uniform(-packet_position_jitter, packet_position_jitter),
        np.random.uniform(-packet_position_jitter, packet_position_jitter)
    ])
    v = np.array([
        300.0 + np.random.uniform(-packet_velocity_jitter, packet_velocity_jitter),
        300.0 + np.random.uniform(-packet_velocity_jitter, packet_velocity_jitter),
        np.random.uniform(-packet_velocity_jitter, packet_velocity_jitter)
    ])
    t = 0.0
    traj = []
    for i in range(n_steps_multi):
        traj.append(r.copy())
        r, v = rk4_step(r, v, t, dt)
        t += dt
    traj = np.array(traj)
    traj_clipped = traj[np.linalg.norm(traj, axis=1) < max_clip_radius]
    if len(traj_clipped) < 2:
        continue
    velocities = np.zeros_like(traj_clipped)
    velocities[1:] = (traj_clipped[1:] - traj_clipped[:-1]) / dt
    disp = traj_clipped[-1] - traj_clipped[0]
    disp_norm = np.linalg.norm(disp)
    if disp_norm == 0:
        continue
    n_hat = disp / disp_norm
    momentum = particle_mass * velocities
    p_parallel = momentum @ n_hat
    dp_parallel_dt = np.diff(p_parallel) / dt
    all_dp_dt.append(dp_parallel_dt)

if all_dp_dt:
    max_len = max(len(arr) for arr in all_dp_dt)
    padded = np.array([np.pad(arr, (0, max_len - len(arr))) for arr in all_dp_dt])
    total_dp_dt = padded.sum(axis=0)
    avg_thrust = np.mean(total_dp_dt)
    print(f"Multi-packet thrust: {avg_thrust:.3e} N")
else:
    print("Multi-packet thrust: no valid packets.")

print("\n===================================================")
print(" NOTEBOOK RUN COMPLETE")
print("===================================================")
